# Tomografía de tiempos y FWI

**Nivel 2 — Análisis aplicado**  
**Prerrequisitos:** nivel 1 y arreglos/logaritmos; 35 min.

## Objetivos

Siga la evidencia desde el contrato fuente hasta el resultado. Declare unidades y supuestos, reproduzca el cálculo e identifique qué inferencia no respalda la evidencia.

## Preparación

Usa módulos y registros versionados desde la raíz descubierta; no descarga ni modifica fuentes. Instale versiones exactas: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Datos

Hay trazas STEAD retenidas y recibos FWI; distinga conteos medidos, FWI sintético, etiquetas humanas y activos web.

## Método

Para fuente `s=(xs,zs)` y receptor `r=(xr,zr)`, control homogéneo `t=||r−s||/v` (m dividido por m·s⁻¹). Sensibilidad tomográfica `δt≈−∫rayo δv/v² ds`; cobertura y trade-off velocidad-profundidad limitan resolución. No están los bytes Koenigsee: no trate el ejemplo upstream como datos locales. FWI minimiza una diferencia como `½||Wd(F(m)−d)||²+regularización`, con velocidad en m/s y operador de onda F. Gradientes adjuntos son habituales; cycle-skipping ocurre si desajuste de fase sale de la cuenca del modelo inicial. Inspeccione verdad, modelos L2/multiescala y WRMS activo/retenido de FWI_LAYERED sintético, en ejes físicos de distancia/profundidad. Es comparación controlada, no validación de campo.

A 100 muestras/s, Nyquist es 50 Hz; respuesta instrumental y preprocesamiento determinan ancho de banda útil. Rayos tomográficos dependen de velocidad actual; FWI usa fase y amplitud completas. Mantenga trazas/disparos retenidos fuera del ajuste. Menor residuo de entrenamiento puede coexistir con peor WRMS retenido o estructura en ciclo equivocado.

## Resultados

Lea ejes físicos y unidades; distinga ajuste, error contra verdad sintética y acuerdo con etiquetas humanas. El recálculo debe concordar con el recibo. Un modelo graficado no demuestra unicidad, transferencia a campo ni resolución.

### Diagnostic 1

Inspeccione el recibo y procedencia del caso.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Inspect the committed synthetic FWI replay and physical grid.
import json
from pathlib import Path
replay=json.loads((root/"data/derived/v2/fwi-replay.json").read_text(encoding="utf-8")); case=json.loads((root/"data/derived/v2/FWI_LAYERED/regularization.json").read_text(encoding="utf-8")); rows=[r for r in replay["records"] if r["case"]=="FWI_LAYERED" and r["variant"]=="reference"]
print("synthetic",case["id"],"grid [x,z]",case["grid"]["shape"],"spacing m",case["grid"]["spacing"],"dt s",case["dt"]); print([(r["method"],r["velocity_rmse"],r["active_wrms"],r["withheld_wrms"]) for r in rows])

### Diagnostic 2

Calcule la comparación y grafíquela con unidades físicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Compare saved truth, L2 and multiscale velocity fields with physical x/depth axes.
import matplotlib.pyplot as plt
nx,nz=case["grid"]["shape"]; dx,dz=case["grid"]["spacing"]; x=np.arange(nx)*dx; z=np.arange(nz)*dz; truth=np.asarray(case["truth"]).reshape(nx,nz).T; fig,ax=plt.subplots(1,3,figsize=(11,3.5),constrained_layout=True)
for a,v,title in zip(ax,[truth]+[np.asarray(case["methods"][k]["model"]).reshape(nx,nz).T for k in ("fwi-l2","fwi-multiscale")],["truth","L2","multiscale"]): im=a.imshow(v,origin="upper",aspect="auto",extent=[x.min(),x.max(),z.max(),z.min()]); a.set(title=title,xlabel="Distance (m)",ylabel="Depth (m)")
fig.colorbar(im,ax=ax,label="Velocity (m/s)"); plt.show(); print("velocity RMSE (m/s), active/withheld WRMS:",[(r["method"],r["velocity_rmse"],r["active_wrms"],r["withheld_wrms"]) for r in rows])

## Resumen

La conclusión depende de geometría, adquisición, ruido, partición, parametrización y derechos de fuente. Conserve negativos y nombre la observación o control siguiente.

## Referencias

- [https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html)
- [https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/](https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/)
- [https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html](https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html)